# Reacher + Mass Değişimi: RL² (SeqPPO) vs PPO Baselines\n
\n
Bu notebook tek hücrelik bir script içerir:\n
\n
- Ortam: **Reacher-v5 (MuJoCo)**, continuous control.\n
- Fiziksel değişiklik: kol kütlesi (body_mass) **mass_scale** ile çarpılıyor.\n
- Baseline 1: **Single-Task PPO (SB3)** – sadece `mass_scale=1.0` ile eğitim.\n
- Baseline 2: **Multi-Task PPO (SB3)** – her reset'te mass_scale rastgele.\n
- Meta yöntem: **RL² + sekans tabanlı PPO (PyTorch)** – farklı mass_scale'ler üzerinde meta-train.\n
\n
En sonda üç yöntemi de aynı mass_skalalarda evaluate edip grafikte karşılaştıracağız.\n
\n
**Not:** Eğitim süreleri uzun olabilir; önce timestep ve meta-iteration sayısını küçültüp debug etmek isteyebilirsin.

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import List, Optional, Callable, Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium import Env

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal
from torch.utils.tensorboard import SummaryWriter

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

plt.rcParams['figure.figsize'] = (7, 4)

# ---------------------------------------------------------------------
# Seed ve cihaz
# ---------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.manual_seed_all(SEED)
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# ---------------------------------------------------------------------
# Mass skala değerleri (hafif -> ağır)
# ---------------------------------------------------------------------
MASS_SCALES = [0.5, 0.75, 1.0, 1.25, 1.5]
print("MASS_SCALES:", MASS_SCALES)

# ---------------------------------------------------------------------
# PPO baseline eğitim süreleri (debug için aşağı çekebilirsin)
# ---------------------------------------------------------------------
TOTAL_TIMESTEPS_SINGLE = 200_000   # Single-Task PPO
TOTAL_TIMESTEPS_MULTI  = 400_000   # Multi-Task PPO

# ---------------------------------------------------------------------
# Meta-RL (RL²) hiperparametreleri
# ---------------------------------------------------------------------
META_BATCH_SIZE   = 4      # her meta-iterasyonda kaç task
EPISODES_PER_TASK = 4      # her task için ardışık epizot sayısı
ROLLOUT_LEN       = 100    # her epizotta max step
TOTAL_META_ITERS  = 200    # meta-iterasyon sayısı
SEG_LEN           = 32     # SeqPPO / TBPTT segment uzunluğu

# PPO / RL² ortak hiperparametreler
GAMMA       = 0.99
LAMBDA_GAE  = 0.95
PPO_EPOCHS  = 4
PPO_CLIP    = 0.2
PPO_LR      = 3e-4
HIDDEN_SIZE = 64

# ---------------------------------------------------------------------
# Reacher gözlem ve aksiyon boyutları
# ---------------------------------------------------------------------
tmp_env = gym.make("Reacher-v5")
obs_dim = tmp_env.observation_space.shape[0]
act_dim = tmp_env.action_space.shape[0]
tmp_env.close()

print("Reacher obs_dim =", obs_dim, "act_dim =", act_dim)

In [ ]:
@dataclass
class MassScaleTask:
    mass_scale: float
    id: int


# ---------------------------------------------------------------------
# SB3 için wrapper (Single-Task & Multi-Task PPO)
# ---------------------------------------------------------------------
class ReacherMassWrapper(gym.Wrapper):
    """
    Reacher-v4 ortamını saran, body_mass'i mass_scale ile çarpan wrapper (SB3 için).
    mode:
      - 'single': sabit mass_scale (fixed_mass_scale)
      - 'multi' : her reset'te rastgele mass_scale
    """
    def __init__(self, env: Env, mode: str,
                 possible_scales: List[float],
                 fixed_mass_scale: Optional[float] = None):
        super().__init__(env)
        assert mode in ["single", "multi"], "mode 'single' veya 'multi' olmalı"

        self.mode = mode
        self.possible_scales = possible_scales
        self.fixed_mass_scale = fixed_mass_scale

        self.model = self.env.unwrapped.model
        self.base_body_mass = self.model.body_mass.copy()
        # World body genelde index 0; onu sabit tutuyoruz
        self.body_indices_to_scale = np.arange(1, self.base_body_mass.shape[0])

        self.current_mass_scale = 1.0

    def _apply_mass_scale(self, scale: float):
        self.current_mass_scale = scale
        self.model.body_mass[:] = self.base_body_mass
        self.model.body_mass[self.body_indices_to_scale] *= scale

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        if self.mode == "single":
            assert self.fixed_mass_scale is not None
            scale = self.fixed_mass_scale
        else:
            scale = random.choice(self.possible_scales)

        self._apply_mass_scale(scale)
        info = dict(info)
        info["mass_scale"] = scale
        return obs.astype(np.float32), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info = dict(info)
        info["mass_scale"] = self.current_mass_scale
        return obs.astype(np.float32), float(reward), bool(terminated), bool(truncated), info


def make_single_task_env(fixed_mass_scale: float = 1.0) -> Callable[[], Env]:
    """Single-Task PPO için env factory (sabit mass_scale)."""
    def _fn():
        base_env = gym.make("Reacher-vv5")
        return ReacherMassWrapper(
            base_env,
            mode="single",
            possible_scales=MASS_SCALES,
            fixed_mass_scale=fixed_mass_scale,
        )
    return _fn


def make_multi_task_env() -> Callable[[], Env]:
    """Multi-Task PPO için env factory (her reset'te random mass_scale)."""
    def _fn():
        base_env = gym.make("Reacher-v5")
        return ReacherMassWrapper(
            base_env,
            mode="multi",
            possible_scales=MASS_SCALES,
            fixed_mass_scale=None,
        )
    return _fn


# ---------------------------------------------------------------------
# RL² tarafı için meta-ortam (SB3'tan bağımsız)
# ---------------------------------------------------------------------
class MetaReacherMassEnv:
    """
    RL² ajanı için kullanılan simple meta-ortam.
    set_task(MassScaleTask) ile mass_scale seçiliyor.
    """
    def __init__(self):
        self.env = gym.make("Reacher-v5")
        self.model = self.env.unwrapped.model
        self.base_body_mass = self.model.body_mass.copy()
        self.body_indices_to_scale = np.arange(1, self.base_body_mass.shape[0])
        self.current_task: Optional[MassScaleTask] = None
        self.current_mass_scale = 1.0

    def _apply_mass_scale(self, scale: float):
        self.current_mass_scale = scale
        self.model.body_mass[:] = self.base_body_mass
        self.model.body_mass[self.body_indices_to_scale] *= scale

    def set_task(self, task: MassScaleTask):
        self.current_task = task
        self._apply_mass_scale(task.mass_scale)

    def reset(self, seed: Optional[int] = None):
        if seed is not None:
            obs, info = self.env.reset(seed=seed)
        else:
            obs, info = self.env.reset()
        if self.current_task is not None:
            self._apply_mass_scale(self.current_task.mass_scale)
        info = dict(info)
        info["mass_scale"] = self.current_mass_scale
        return obs.astype(np.float32), info

    def step(self, action: np.ndarray):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info = dict(info)
        info["mass_scale"] = self.current_mass_scale
        return obs.astype(np.float32), float(reward), bool(terminated), bool(truncated), info

    def close(self):
        self.env.close()


def sample_mass_task_for_meta(task_id: int) -> MassScaleTask:
    scale = random.choice(MASS_SCALES)
    return MassScaleTask(mass_scale=scale, id=task_id)

In [ ]:
# ---------------------------------------------------------------------
# PPO baseline eğitim fonksiyonları (SB3)
# ---------------------------------------------------------------------
def train_single_task_ppo(
    total_timesteps: int,
    fixed_mass_scale: float = 1.0,
    n_envs: int = 8,
    log_dir: str = "./logs_single_task",
) -> PPO:
    os.makedirs(log_dir, exist_ok=True)

    env_fn = make_single_task_env(fixed_mass_scale=fixed_mass_scale)
    env = make_vec_env(env_fn, n_envs=n_envs, seed=SEED)

    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        seed=SEED,
        tensorboard_log=log_dir,
        n_steps=2048,
        batch_size=64,
        gae_lambda=0.95,
        gamma=0.99,
        n_epochs=10,
        clip_range=0.2,
        learning_rate=3e-4,
    )

    model.learn(total_timesteps=total_timesteps)
    env.close()
    return model


def train_multi_task_ppo(
    total_timesteps: int,
    n_envs: int = 8,
    log_dir: str = "./logs_multi_task",
) -> PPO:
    os.makedirs(log_dir, exist_ok=True)

    env_fn = make_multi_task_env()
    env = make_vec_env(env_fn, n_envs=n_envs, seed=SEED)

    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        seed=SEED,
        tensorboard_log=log_dir,
        n_steps=2048,
        batch_size=64,
        gae_lambda=0.95,
        gamma=0.99,
        n_epochs=10,
        clip_range=0.2,
        learning_rate=3e-4,
    )

    model.learn(total_timesteps=total_timesteps)
    env.close()
    return model


# ---------------------------------------------------------------------
# PPO evaluate fonksiyonu
# ---------------------------------------------------------------------
def evaluate_ppo_on_mass_scales(
    model: PPO,
    mass_scales: List[float],
    n_eval_episodes: int = 10,
    deterministic: bool = True,
) -> Dict[float, Tuple[float, float]]:
    results: Dict[float, Tuple[float, float]] = {}
    for scale in mass_scales:
        env_fn = make_single_task_env(fixed_mass_scale=scale)
        eval_env = env_fn()

        mean_reward, std_reward = evaluate_policy(
            model,
            eval_env,
            n_eval_episodes=n_eval_episodes,
            deterministic=deterministic,
            render=False,
        )

        results[scale] = (float(mean_reward), float(std_reward))
        eval_env.close()

        print(
            f"[PPO EVAL] mass_scale={scale:.2f} | "
            f"mean_return={mean_reward:.2f} ± {std_reward:.2f}"
        )

    return results

In [ ]:
# ---------------------------------------------------------------------
# RL² girdi: augmented observation
#   aug_t = [obs_t, a_{t-1}, r_{t-1}, d_{t-1}]
# ---------------------------------------------------------------------
def make_initial_history(action_dim: int):
    a_prev = np.zeros((action_dim,), dtype=np.float32)
    r_prev = np.array([0.0], dtype=np.float32)
    d_prev = np.array([0.0], dtype=np.float32)
    return a_prev, r_prev, d_prev


def augment_obs(obs, a_prev, r_prev, d_prev):
    return np.concatenate([obs, a_prev, r_prev, d_prev], axis=-1).astype(np.float32)


# Augmented obs dim kontrolü
_tmp_env = MetaReacherMassEnv()
_tmp_task = sample_mass_task_for_meta(0)
_tmp_env.set_task(_tmp_task)
s0, _ = _tmp_env.reset(seed=SEED)
a0, r0, d0 = make_initial_history(act_dim)
aug0 = augment_obs(s0, a0, r0, d0)
_tmp_env.close()
aug_dim = aug0.shape[0]
print("Augmented obs dim =", aug_dim)


# ---------------------------------------------------------------------
# RL² Policy (GRU tabanlı)
# ---------------------------------------------------------------------
class RL2RecurrentPolicy(nn.Module):
    def __init__(self, input_dim: int, action_dim: int, hidden_size: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.action_dim  = action_dim

        self.fc_in = nn.Linear(input_dim, hidden_size)
        self.gru   = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.fc_pi   = nn.Linear(hidden_size, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.fc_v    = nn.Linear(hidden_size, 1)

    def initial_hidden(self, batch_size: int = 1, device: str = "cpu"):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

    def forward_step(self, x_t: torch.Tensor, h: torch.Tensor):
        # x_t: (B, input_dim), h: (1, B, H)
        z = torch.relu(self.fc_in(x_t))
        z = z.unsqueeze(1)          # (B, 1, H)
        out, h_next = self.gru(z, h)
        out = out.squeeze(1)        # (B, H)

        mean = self.fc_pi(out)
        std  = self.log_std.exp().unsqueeze(0)
        dist = Normal(mean, std)
        action = dist.sample()
        logp   = dist.log_prob(action).sum(dim=-1)
        value  = self.fc_v(out).squeeze(-1)
        return action, logp, value, h_next

    def act_deterministic(self, x_t: torch.Tensor, h: torch.Tensor):
        z = torch.relu(self.fc_in(x_t))
        z = z.unsqueeze(1)
        out, h_next = self.gru(z, h)
        out = out.squeeze(1)
        mean  = self.fc_pi(out)
        value = self.fc_v(out).squeeze(-1)
        return mean, value, h_next


# Küçük debug
policy_dbg = RL2RecurrentPolicy(input_dim=aug_dim, action_dim=act_dim, hidden_size=HIDDEN_SIZE).to(device)
h0_dbg = policy_dbg.initial_hidden(batch_size=1, device=device)
x_dbg = torch.tensor(aug0, dtype=torch.float32, device=device).unsqueeze(0)
a_dbg, logp_dbg, v_dbg, h1_dbg = policy_dbg.forward_step(x_dbg, h0_dbg)
print("Policy debug action shape:", a_dbg.shape, "value:", v_dbg.item())


# ---------------------------------------------------------------------
# RL² Agent (collect_batch + SeqPPO + return normalizasyonu)
# ---------------------------------------------------------------------
class RL2Agent:
    def __init__(self, input_dim, action_dim, hidden_size=HIDDEN_SIZE, lr=PPO_LR, device=device):
        self.device = device
        self.policy = RL2RecurrentPolicy(input_dim, action_dim, hidden_size).to(device)
        self.optim  = torch.optim.Adam(self.policy.parameters(), lr=lr)

    def collect_batch(
        self,
        meta_batch_size=META_BATCH_SIZE,
        episodes_per_task=EPISODES_PER_TASK,
        rollout_len=ROLLOUT_LEN,
    ):
        obs_buf, act_buf, logp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []
        z_buf, task_id_buf, task_start_buf = [], [], []

        ep_returns_per_task = []

        for task_idx in range(meta_batch_size):
            task = sample_mass_task_for_meta(task_id=task_idx)
            env  = MetaReacherMassEnv()
            env.set_task(task)

            h = self.policy.initial_hidden(batch_size=1, device=self.device)
            first_step_of_task = True
            task_ep_returns = []

            for ep in range(episodes_per_task):
                s, _ = env.reset(seed=SEED + task_idx * 100 + ep)
                prev_a, prev_r, prev_d = make_initial_history(act_dim)
                ep_ret = 0.0

                for t in range(rollout_len):
                    aug = augment_obs(s, prev_a, prev_r, prev_d)
                    x_t = torch.tensor(aug, dtype=torch.float32, device=self.device).unsqueeze(0)
                    with torch.no_grad():
                        a_t, logp_t, v_t, h = self.policy.forward_step(x_t, h)

                    a_np = a_t.squeeze(0).cpu().numpy()
                    ns, r, done, trunc, info = env.step(a_np)

                    if first_step_of_task:
                        task_start_buf.append(1.0)
                        first_step_of_task = False
                    else:
                        task_start_buf.append(0.0)

                    obs_buf.append(aug)
                    act_buf.append(a_np)
                    logp_buf.append(logp_t.cpu().numpy())
                    rew_buf.append(r)
                    val_buf.append(v_t.cpu().numpy())
                    done_flag = float(done or trunc)
                    done_buf.append(done_flag)

                    z_buf.append(h.squeeze(0).squeeze(0).detach().cpu().numpy())
                    task_id_buf.append(task.id)

                    ep_ret += r

                    prev_a = a_np.astype(np.float32)
                    prev_r = np.array([r], dtype=np.float32)
                    prev_d = np.array([done_flag], dtype=np.float32)
                    s = ns
                    if done or trunc:
                        break

                task_ep_returns.append(ep_ret)

            ep_returns_per_task.append(task_ep_returns)
            env.close()

        batch = {
            "obs":        torch.tensor(np.stack(obs_buf), dtype=torch.float32, device=self.device),
            "act":        torch.tensor(np.stack(act_buf), dtype=torch.float32, device=self.device),
            "logp":       torch.tensor(np.stack(logp_buf).squeeze(-1), dtype=torch.float32, device=self.device),
            "rew":        torch.tensor(np.array(rew_buf), dtype=torch.float32, device=self.device),
            "val":        torch.tensor(np.stack(val_buf).squeeze(-1), dtype=torch.float32, device=self.device),
            "done":       torch.tensor(np.array(done_buf), dtype=torch.float32, device=self.device),
            "z":          torch.tensor(np.stack(z_buf), dtype=torch.float32, device=self.device),
            "task_id":    torch.tensor(np.array(task_id_buf), dtype=torch.float32, device=self.device),
            "task_start": torch.tensor(np.array(task_start_buf), dtype=torch.float32, device=self.device),
        }

        return batch, ep_returns_per_task

    def compute_gae(self, rew, val, done, gamma=GAMMA, lam=LAMBDA_GAE):
        T = len(rew)
        adv = torch.zeros(T, dtype=torch.float32, device=rew.device)
        last_gae = 0.0
        for t in reversed(range(T)):
            nonterminal = 1.0 - done[t]
            if t == T - 1:
                next_value = 0.0
            else:
                next_value = val[t + 1]
            delta = rew[t] + gamma * next_value * nonterminal - val[t]
            last_gae = delta + gamma * lam * nonterminal * last_gae
            adv[t] = last_gae
        ret = adv + val
        return adv, ret

    def ppo_update(self, batch, epochs=PPO_EPOCHS, clip_ratio=PPO_CLIP, seg_len=SEG_LEN):
        obs        = batch["obs"]
        act        = batch["act"]
        old_logp   = batch["logp"]
        rew        = batch["rew"]
        val        = batch["val"]
        done       = batch["done"]
        task_start = batch["task_start"]

        adv, ret = self.compute_gae(rew, val, done)
        # Advantage normalizasyonu
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        # Return normalizasyonu (hocanın övdüğü hack)
        ret = (ret - ret.mean()) / (ret.std() + 1e-8)

        T = obs.shape[0]
        last_pi_loss = 0.0
        last_v_loss  = 0.0

        for ep in range(epochs):
            h = self.policy.initial_hidden(batch_size=1, device=self.device)
            t = 0
            while t < T:
                seg_end = min(t + seg_len, T)

                self.optim.zero_grad()
                seg_policy_losses = []
                seg_value_losses  = []
                seg_entropies     = []

                while t < seg_end:
                    if task_start[t] > 0.5:
                        h = self.policy.initial_hidden(batch_size=1, device=self.device)

                    x_t        = obs[t].unsqueeze(0)
                    a_t        = act[t].unsqueeze(0)
                    adv_t      = adv[t].unsqueeze(0)
                    ret_t      = ret[t].unsqueeze(0)
                    old_logp_t = old_logp[t].unsqueeze(0)

                    mean_t, v_t, h = self.policy.act_deterministic(x_t, h)
                    dist  = Normal(mean_t, self.policy.log_std.exp().unsqueeze(0))
                    logp_t = dist.log_prob(a_t).sum(dim=-1)

                    ratio = torch.exp(logp_t - old_logp_t)
                    surr1 = ratio * adv_t
                    surr2 = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_t
                    policy_loss_t = -torch.min(surr1, surr2).mean()
                    value_loss_t  = F.mse_loss(v_t, ret_t)
                    entropy_t     = dist.entropy().sum(dim=-1).mean()

                    seg_policy_losses.append(policy_loss_t)
                    seg_value_losses.append(value_loss_t)
                    seg_entropies.append(entropy_t)

                    t += 1

                if len(seg_policy_losses) == 0:
                    break

                policy_loss = torch.stack(seg_policy_losses).mean()
                value_loss  = torch.stack(seg_value_losses).mean()
                entropy     = torch.stack(seg_entropies).mean()

                loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
                self.optim.step()

                h = h.detach()

                last_pi_loss = float(policy_loss.item())
                last_v_loss  = float(value_loss.item())

        return last_pi_loss, last_v_loss


meta_agent = RL2Agent(input_dim=aug_dim, action_dim=act_dim, hidden_size=HIDDEN_SIZE, lr=PPO_LR, device=device)
print("RL² meta_agent hazır.")

In [ ]:
rl2_avg_return_history = []
rl2_last_ep_return_history = []
rl2_pi_loss_history = []
rl2_v_loss_history  = []

writer_rl2 = SummaryWriter(log_dir="./logs_rl2")

for it in range(1, TOTAL_META_ITERS + 1):
    batch, ep_returns_per_task = meta_agent.collect_batch()
    pi_loss, v_loss = meta_agent.ppo_update(batch)

    flat_returns = []
    last_eps = []
    for eps in ep_returns_per_task:
        if len(eps) > 0:
            flat_returns.extend(eps)
            last_eps.append(eps[-1])

    avg_ret = float(np.mean(flat_returns)) if len(flat_returns) > 0 else 0.0
    last_ep_ret = float(np.mean(last_eps)) if len(last_eps) > 0 else 0.0

    rl2_avg_return_history.append(avg_ret)
    rl2_last_ep_return_history.append(last_ep_ret)
    rl2_pi_loss_history.append(pi_loss)
    rl2_v_loss_history.append(v_loss)

    # TensorBoard log
    writer_rl2.add_scalar("meta/avg_return", avg_ret, it)
    writer_rl2.add_scalar("meta/last_ep_return", last_ep_ret, it)
    writer_rl2.add_scalar("loss/policy", pi_loss, it)
    writer_rl2.add_scalar("loss/value", v_loss, it)

    if it % 10 == 0:
        print(
            f"[RL²] Iter {it:04d} | avg_ret={avg_ret:.3f} | "
            f"last_ep_ret={last_ep_ret:.3f} | pi_loss={pi_loss:.3f} | v_loss={v_loss:.3f}"
        )

writer_rl2.close()
print("RL² training complete.")

In [ ]:
@torch.no_grad()
def evaluate_rl2_on_mass_scales(
    agent: RL2Agent,
    mass_scales: List[float],
    n_eval_episodes: int = 10,
    rollout_len: int = 200,
) -> Dict[float, Tuple[float, float]]:
    results: Dict[float, Tuple[float, float]] = {}
    for scale in mass_scales:
        returns = []
        for ep in range(n_eval_episodes):
            task = MassScaleTask(mass_scale=scale, id=0)
            env = MetaReacherMassEnv()
            env.set_task(task)

            s, _ = env.reset(seed=SEED + ep)
            prev_a, prev_r, prev_d = make_initial_history(act_dim)
            h = agent.policy.initial_hidden(batch_size=1, device=agent.device)

            ep_ret = 0.0
            for t in range(rollout_len):
                aug = augment_obs(s, prev_a, prev_r, prev_d)
                x_t = torch.tensor(aug, dtype=torch.float32, device=agent.device).unsqueeze(0)
                mean_t, v_t, h = agent.policy.act_deterministic(x_t, h)
                a_t = mean_t.squeeze(0).cpu().numpy()
                ns, r, done, trunc, info = env.step(a_t)

                ep_ret += r

                prev_a = a_t.astype(np.float32)
                prev_r = np.array([r], dtype=np.float32)
                prev_d = np.array([float(done or trunc)], dtype=np.float32)
                s = ns
                if done or trunc:
                    break

            env.close()
            returns.append(ep_ret)

        mean_ret = float(np.mean(returns))
        std_ret  = float(np.std(returns))
        results[scale] = (mean_ret, std_ret)
        print(
            f"[RL² EVAL] mass_scale={scale:.2f} | "
            f"mean_return={mean_ret:.2f} ± {std_ret:.2f}"
        )

    return results


def plot_comparison(single_results, multi_results, rl2_results, mass_scales):
    ms = sorted(mass_scales)
    single_means = [single_results[m][0] for m in ms]
    multi_means  = [multi_results[m][0]  for m in ms]
    rl2_means    = [rl2_results[m][0]    for m in ms]

    plt.figure(figsize=(7, 4))
    plt.plot(ms, single_means, marker="o", label="Single-Task PPO")
    plt.plot(ms, multi_means,  marker="s", label="Multi-Task PPO")
    plt.plot(ms, rl2_means,    marker="^", label="RL² (Meta)")
    plt.xlabel("mass_scale")
    plt.ylabel("Mean Return")
    plt.title("Reacher + Mass Değişimi: Yöntem Karşılaştırması")
    plt.grid(True)
    plt.legend()
    plt.show()


# -------------------------------------------------------------
# Şimdi PPO baselinelerini eğit + hepsini evaluate et
# (İstersen ilk denemede timesteps'leri çok daha düşük tut)
# -------------------------------------------------------------

print("\n>>> Training Single-Task PPO (mass_scale = 1.0)")
single_model = train_single_task_ppo(
    total_timesteps=TOTAL_TIMESTEPS_SINGLE,
    fixed_mass_scale=1.0,
    n_envs=8,
    log_dir="./logs_single_task",
)

print("\n>>> Evaluating Single-Task PPO on MASS_SCALES")
single_results = evaluate_ppo_on_mass_scales(
    single_model,
    MASS_SCALES,
    n_eval_episodes=10,
)

print("\n>>> Training Multi-Task PPO (mass_scale random per episode)")
multi_model = train_multi_task_ppo(
    total_timesteps=TOTAL_TIMESTEPS_MULTI,
    n_envs=8,
    log_dir="./logs_multi_task",
)

print("\n>>> Evaluating Multi-Task PPO on MASS_SCALES")
multi_results = evaluate_ppo_on_mass_scales(
    multi_model,
    MASS_SCALES,
    n_eval_episodes=10,
)

print("\n>>> Evaluating RL² Meta-Agent on MASS_SCALES")
rl2_results = evaluate_rl2_on_mass_scales(
    meta_agent,
    MASS_SCALES,
    n_eval_episodes=10,
    rollout_len=200,
)

print("\nSingle-Task PPO Results:", single_results)
print("Multi-Task PPO Results:", multi_results)
print("RL² Results:", rl2_results)

plot_comparison(single_results, multi_results, rl2_results, MASS_SCALES)

print("\nTensorBoard için:")
print("  tensorboard --logdir logs_single_task,logs_multi_task,logs_rl2")

In [ ]:
def evaluate_dynamic_mass_change(agent, ppo_model, rollout_len=200, change_step=25):
    """
    İlk yarıda mass=0.5, ikinci yarıda mass=2.0 olan bir senaryo.
    """
    # Ortamı hazırla
    env = MetaReacherMassEnv()
    
    # --- RL² Testi ---
    task_start = MassScaleTask(mass_scale=0.5, id=1)
    task_end   = MassScaleTask(mass_scale=2.0, id=2) # Eğitimde görülmemiş bir ağırlık!
    
    env.set_task(task_start)
    s, _ = env.reset(seed=SEED)
    
    prev_a, prev_r, prev_d = make_initial_history(act_dim)
    h = agent.policy.initial_hidden(batch_size=1, device=agent.device)
    
    rl2_rewards = []
    
    for t in range(rollout_len):
        # Dinamik değişim anı
        if t == change_step:
            print(f"!!! Değişim Anı: Step {t} -> Mass 2.0 !!!")
            env.set_task(task_end) # Ortamın fiziğini değiştir
            
        # RL² Step
        aug = augment_obs(s, prev_a, prev_r, prev_d)
        x_t = torch.tensor(aug, dtype=torch.float32, device=agent.device).unsqueeze(0)
        with torch.no_grad():
            mean_t, v_t, h = agent.policy.act_deterministic(x_t, h)
        a_t = mean_t.squeeze(0).cpu().numpy()
        
        ns, r, done, trunc, info = env.step(a_t)
        rl2_rewards.append(r)
        
        prev_a = a_t
        prev_r = np.array([r], dtype=np.float32)
        prev_d = np.array([float(done)], dtype=np.float32)
        s = ns
        if done or trunc: break
            
    env.close()

    # --- Multi-Task PPO Testi (Kıyaslama için) ---
    # Not: SB3 modelini bu 'MetaReacherMassEnv' üzerinde manuel step ile çalıştıracağız
    env = MetaReacherMassEnv()
    env.set_task(task_start)
    s, _ = env.reset(seed=SEED)
    
    ppo_rewards = []
    for t in range(rollout_len):
        if t == change_step:
            env.set_task(task_end)
            
        # PPO sadece observation alır (augmented değil)
        action, _ = ppo_model.predict(s, deterministic=True)
        s, r, done, trunc, _ = env.step(action)
        ppo_rewards.append(r)
        if done or trunc: break
    env.close()
    
    # --- Grafiği Çiz ---
    plt.figure(figsize=(10, 5))
    plt.plot(rl2_rewards, label="RL² (Meta)", alpha=0.8)
    plt.plot(ppo_rewards, label="Multi-Task PPO", alpha=0.8, linestyle="--")
    plt.axvline(x=change_step, color='r', linestyle=':', label="Mass Change")
    plt.title("Dynamic Adaptation Test: Mass 0.5 -> 2.0")
    plt.xlabel("Step")
    plt.ylabel("Reward")
    plt.legend()
    plt.show()

# Kullanımı:
evaluate_dynamic_mass_change(meta_agent, multi_model)

PYGAME

In [ ]:
import pygame
import numpy as np
import torch
import gymnasium as gym
import time

def play_with_pygame(agent_type, model, mass_scale, rollout_len=200):
    """
    Ajanı belirtilen kütle ölçeğinde (mass_scale) çalıştırır ve Pygame ile gösterir.
    agent_type: 'ppo' veya 'rl2'
    """
    # 1. Ortamı 'rgb_array' render moduyla oluştur
    # Not: Render modu görseli array olarak almamızı sağlar
    render_env = gym.make("Reacher-v5", render_mode="rgb_array", max_episode_steps=rollout_len)
    
    # --- Ortamın Kütlesini Ayarla (Wrapper Mantığı) ---
    base_mass = render_env.unwrapped.model.body_mass.copy()
    body_indices = np.arange(1, base_mass.shape[0]) # World harici body'ler
    render_env.unwrapped.model.body_mass[:] = base_mass
    render_env.unwrapped.model.body_mass[body_indices] *= mass_scale
    # --------------------------------------------------

    # Pygame Başlatma (İlk seferde)
    pygame.init()
    screen_width, screen_height = 500, 500 # Reacher varsayılan render boyutu genelde 500x500'dür
    screen = pygame.display.set_mode((screen_width, screen_height))
    pygame.display.set_caption(f"Reacher Vis | Agent: {agent_type} | Mass: {mass_scale}")
    clock = pygame.time.Clock()
    font = pygame.font.SysFont("Arial", 18)

    # Reset
    s, _ = render_env.reset(seed=SEED)
    
    # RL2 için hafıza ilklendirme
    if agent_type == "rl2":
        prev_a, prev_r, prev_d = make_initial_history(act_dim)
        # Modeli ve hidden state'i hazırla
        device = model.device
        h = model.policy.initial_hidden(batch_size=1, device=device)

    total_reward = 0.0
    
    running = True
    for t in range(rollout_len):
        # Pygame eventlerini kontrol et (Pencere kapatma vb.)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                break
        if not running: break

        # --- Render Görüntüsünü Al ve Pygame'e Bas ---
        frame = render_env.render()  # (H, W, 3) RGB Array döner
        # Frame'i Pygame surface'ına çevir (Transpose ve Axis değişimi gerekebilir)
        # MuJoCo genelde (Height, Width, 3) döner. Pygame (Width, Height) ister.
        # Ayrıca image'ı 90 derece döndürmek gerekebilir.
        frame_surface = pygame.surfarray.make_surface(np.transpose(frame, (1, 0, 2)))
        # Koordinat sistemi farkından dolayı genelde flip/rotate gerekir, deneme-yanılma ile:
        frame_surface = pygame.transform.rotate(frame_surface, -90)
        frame_surface = pygame.transform.flip(frame_surface, True, False)
        
        screen.blit(frame_surface, (0, 0))

        # --- Bilgi Yazıları (UI) ---
        text_agent = font.render(f"Agent: {agent_type.upper()}", True, (255, 255, 255))
        text_mass  = font.render(f"Mass Scale: {mass_scale:.1f}x", True, (255, 255, 0)) # Sarı renk
        text_step  = font.render(f"Step: {t}/{rollout_len}", True, (255, 255, 255))
        text_rew   = font.render(f"Return: {total_reward:.1f}", True, (0, 255, 0)) # Yeşil renk
        
        screen.blit(text_agent, (10, 10))
        screen.blit(text_mass, (10, 30))
        screen.blit(text_step, (10, 50))
        screen.blit(text_rew, (10, 70))
        
        pygame.display.flip()
        clock.tick(30) # 30 FPS sınırı

        # --- Aksiyon Seçimi ---
        if agent_type == "ppo":
            # SB3 modeli
            action, _ = model.predict(s, deterministic=True)
        
        elif agent_type == "rl2":
            # RL2 modeli (Hidden state yönetimi)
            aug = augment_obs(s, prev_a, prev_r, prev_d)
            x_t = torch.tensor(aug, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                mean_t, v_t, h = model.policy.act_deterministic(x_t, h)
            action = mean_t.squeeze(0).cpu().numpy()

        # Adım at
        ns, r, done, trunc, _ = render_env.step(action)
        total_reward += r

        # RL2 için geçmişi güncelle
        if agent_type == "rl2":
            prev_a = action.astype(np.float32)
            prev_r = np.array([r], dtype=np.float32)
            prev_d = np.array([float(done or trunc)], dtype=np.float32)

        s = ns
        if done or trunc:
            break
            
    render_env.close()
    pygame.quit()
    print(f"Episode finished. Agent: {agent_type}, Return: {total_reward:.2f}")


# ==========================================
# GÖRSELLEŞTİRME SENARYOSU
# ==========================================

# 1. Baseline PPO'yu Test Et (Ağır bir kütle ile: 1.5x)
print(">>> Gösterim 1: Multi-Task PPO (Mass 1.5x)")
play_with_pygame(agent_type="ppo", model=multi_model, mass_scale=1.5)

# Kısa bir bekleme
time.sleep(1)

# 2. RL2 Meta Agent'ı Test Et (Aynı ağır kütle ile: 1.5x)
# Farkı görmek için: RL2 başta zorlanıp sonra hedefe kitlenmeli.
print(">>> Gösterim 2: RL² Meta Agent (Mass 1.5x)")
play_with_pygame(agent_type="rl2", model=meta_agent, mass_scale=1.5)

# 3. (Opsiyonel) Çok uç bir durum (Mass 3.0x - Çok Ağır)
# RL2'nin adaptasyonunu burada daha net görebilirsin.
print(">>> Gösterim 3: RL² Meta Agent (Extreme Mass 3.0x)")
play_with_pygame(agent_type="rl2", model=meta_agent, mass_scale=3.0)